In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content
!rm -rf PIDNet  # Ensures a clean start
!git clone https://github.com/XuJiacong/PIDNet.git
%cd /content/PIDNet

/content
Cloning into 'PIDNet'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 386 (delta 131), reused 125 (delta 125), pack-reused 193 (from 1)
Receiving objects: 100% (386/386), 212.80 MiB | 59.86 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/content/PIDNet


In [ ]:
!cp -f /content/drive/MyDrive/CSS/model/pidnet_cl.py /content/PIDNet/models/pidnet_cl.py

Load Model

In [ ]:
import torch
import torch.nn as nn
from models.pidnet_cl import PIDNetCL

# 1. Define the Robust Loader Function
def load_checkpoint_strictish(model, ckpt_path):
    print(f"=> Loading checkpoint from: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cpu")

    # A) Get the actual state dict
    if isinstance(ckpt, dict):
        for k in ["state_dict", "model", "model_state", "net", "ema_state_dict"]:
            if k in ckpt and isinstance(ckpt[k], dict):
                ckpt = ckpt[k]
                break

    # B) Strip common prefixes (handle DataParallel/DDP wrappers)
    def strip_prefix(s):
        for p in ["module.model.", "module.", "model.", "net.", "seg_model."]:
            if s.startswith(p):
                return s[len(p):]
        return s

    raw = {strip_prefix(k): v for k, v in ckpt.items() if isinstance(v, torch.Tensor)}

    # C) Keep only matching keys (same name and shape)
    cur = model.state_dict()
    filt = {k: v for k, v in raw.items() if k in cur and v.shape == cur[k].shape}

    missing, unexpected = model.load_state_dict(filt, strict=False)
    print(f"   Loaded {len(filt)} tensors | Missing {len(missing)} | Unexpected {len(unexpected)}")
    return missing, unexpected

# 2. Initialize Model
print("=> Initializing PIDNetCL...")
model = PIDNetCL(m=2, n=3, num_classes=15, planes=32, ppm_planes=96, head_planes=128, augment=True)

# 3. Load Pretrained Weights (Base 15 Classes)
checkpoint_path = "/content/drive/MyDrive/CSS/PIDNet/output/cityscapes/pidnet_s_custom_15/best.pt"
missing, unexpected = load_checkpoint_strictish(model, checkpoint_path)

# 4. Verify Loading (Sanity Check)
print("\n=> Verifying Weights:")
# Check if missing keys are ONLY the new branches (which is what we want)
expected_missing_prefixes = ("layer3_d_new", "layer4_d_new", "diff3_new", "diff4_new", "seghead_new")
is_clean_loading = all(k.startswith(expected_missing_prefixes) for k in missing)

if is_clean_loading and len(unexpected) == 0:
    print("   SUCCESS: Base model loaded correctly. Only new CL branches are uninitialized.")
else:
    print("   WARNING: Unexpected keys found! Check your model architecture.")
    print("   First 10 missing:", missing[:10])

# 5. Verify Forward Pass (Shape Check)
print("\n=> Checking Forward Pass Shapes:")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    # Create dummy input (B, C, H, W)
    x = torch.randn(1, 3, 512, 1024).to(device)
    out = model.forward_with_new(x)

print(f"   Old Head Output: {out['logits_15'].shape} (Expected: [1, 15, 512, 1024])")
print(f"   New Head Output: {out['logits_new_2ch'].shape} (Expected: [1, 2, 512, 1024])")

=> Initializing PIDNetCL...
=> Loading checkpoint from: /content/drive/MyDrive/CSS/PIDNet/output/cityscapes/pidnet_s_custom_15/best.pt
   Loaded 479 tensors | Missing 56 | Unexpected 0

=> Verifying Weights:
   SUCCESS: Base model loaded correctly. Only new CL branches are uninitialized.

=> Checking Forward Pass Shapes:
   Old Head Output: torch.Size([1, 15, 512, 1024]) (Expected: [1, 15, 512, 1024])
   New Head Output: torch.Size([1, 2, 512, 1024]) (Expected: [1, 2, 512, 1024])


Load Dataset

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/CSS/PIDNet/data/cityscapes"
NEW_CLASS_ORIG_ID = 28  # 28=Bus
IGNORE_ID = 255

import os, glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import numpy as np

# Standard ImageNet normalization
img_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

class CityscapesBinaryNew(Dataset):
    """
    Reads Cityscapes images, converts RGB -> BGR to match the original
    PIDNet backbone training (which used cv2.imread), and creates binary masks.
    """
    def __init__(self, root, split="train", new_class_orig_id=NEW_CLASS_ORIG_ID):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir  = os.path.join(root, "gtFine", split)
        self.new_class_orig_id = new_class_orig_id

        self.items = []
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                city_img_dir = os.path.join(self.img_dir, city)
                imgs = sorted(glob.glob(os.path.join(city_img_dir, "*_leftImg8bit.png")))
                for ip in imgs:
                    base_name = os.path.basename(ip).replace("_leftImg8bit.png", "")
                    gt_name = base_name + "_gtFine_labelIds.png"
                    gp = os.path.join(self.gt_dir, city, gt_name)
                    if os.path.exists(gp):
                        self.items.append((ip, gp))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ip, gp = self.items[idx]

        # 1. Load Image (PIL loads as RGB)
        img_pil = Image.open(ip).convert("RGB")
        img_np = np.array(img_pil)

        # 2. CRITICAL FIX: Convert RGB -> BGR
        # This ensures the 'Blue' channel is in position 0, matching cv2.imread
        # and the backbone's expectations.
        img_bgr = img_np[:, :, ::-1].copy()

        # 3. Transform (Normalizes the BGR image using the provided stats)
        img_t = img_tf(img_bgr)

        # 4. Load Label
        lbl = Image.open(gp)
        lbl_np = np.array(lbl, dtype=np.uint8)

        # 5. Create Binary Mask
        tgt = np.full_like(lbl_np, 255, dtype=np.uint8)

        # Set Old Classes to 0 (Background)
        valid_old_classes = [7, 8, 11, 12, 13, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27]
        for c in valid_old_classes:
            tgt[lbl_np == c] = 0

        # Set New Class to 1 (Target)
        tgt[lbl_np == self.new_class_orig_id] = 1

        bin_t = torch.from_numpy(tgt).long()

        return img_t, bin_t

# Re-build loaders
train_set = CityscapesBinaryNew(DATA_ROOT, "train")
val_set   = CityscapesBinaryNew(DATA_ROOT, "val")

if len(train_set) > 0:
    train_loader = DataLoader(train_set, batch_size=2, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_set,   batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
    print("Dataset re-loaded with BGR Fix.")

Dataset re-loaded with BGR Fix.


Train

In [ ]:
model.train()

# Train only the new branch & head parameters
for name, p in model.named_parameters():
    # Only unfreeze if the name contains "_new"
    p.requires_grad = any(t in name for t in ["layer3_d_new","layer4_d_new","diff3_new","diff4_new","seghead_new"])

# OVERRIDE: Keep frozen base BN in eval; let new-branch BN train
# (Must happen AFTER model.train())
for name, m in model.named_modules():
    if isinstance(m, nn.BatchNorm2d):
        if any(t in name for t in ["_new", "seghead_new"]):
            # New branches: Learn new stats
            m.train()
        else:
            # Base model: Freeze stats (use pre-trained mean/var)
            m.eval()
            m.track_running_stats = False

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model configured for training:")
print(f"- Mode: { 'CUDA' if torch.cuda.is_available() else 'CPU' }")
print("- Base BN layers frozen? Yes")
print("- Gradients active only for '_new' layers? Yes")

Model configured for training:
- Mode: CUDA
- Base BN layers frozen? Yes
- Gradients active only for '_new' layers? Yes


In [ ]:
optim = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-3, weight_decay=1e-4)
crit  = nn.CrossEntropyLoss(ignore_index=255)

In [ ]:
# --- TRAIN: Scan FULL dataset for frames that contain class 28 (Bus) ---
import numpy as np
from PIL import Image
from torch.utils.data import Subset, DataLoader

# Set the correct ID for "Bus"
TARGET_ID = 28

print(f"Scanning full dataset for images containing class {TARGET_ID}...")

train_pos_indices = []
# Loop through the ENTIRE dataset to find every single bus
for i, (_, gt_path) in enumerate(train_set.items):
    lbl = np.array(Image.open(gt_path), dtype=np.uint8)
    if (lbl == TARGET_ID).any():
        train_pos_indices.append(i)

print(f"Found {len(train_pos_indices)} useful images out of {len(train_set)} total.")

# Create the Subset
train_subset = Subset(train_set, train_pos_indices)

# Create the Loader
# CRITICAL: shuffle=True is needed for serious training
train_loader_pos = DataLoader(
    train_subset,
    batch_size=2,
    shuffle=True,   # Randomize order for better convergence
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

# Sanity Check
with torch.no_grad():
    imgs, lbls = next(iter(train_loader_pos))
    pos = (lbls == 1).sum().item()
    tot = (lbls != 255).sum().item()
    print("------------------------------------------------")
    print(f"Batch loaded successfully.")
    print(f"Contains target pixels? {'Yes' if pos > 0 else 'No'}")
    print(f"Positive pixels in this batch: {pos}")

Scanning full dataset for images containing class 28...
Found 274 useful images out of 2975 total.
------------------------------------------------
Batch loaded successfully.
Contains target pixels? Yes
Positive pixels in this batch: 169879


In [ ]:
# --- VAL: Scan VALIDATION dataset for frames that contain class 28 (Bus) ---
import numpy as np
from PIL import Image
from torch.utils.data import Subset, SequentialSampler, DataLoader

# Ensure we look for the same ID as training (28)
# If TARGET_ID isn't defined from the previous cell, we set it here.
if 'TARGET_ID' not in locals():
    TARGET_ID = 28

print(f"Scanning validation dataset for images containing class {TARGET_ID}...")

val_pos_indices = []
for i, (_, gt_path) in enumerate(val_set.items):
    lbl = np.array(Image.open(gt_path), dtype=np.uint8)

    # Check for Bus pixels
    if (lbl == TARGET_ID).any():
        val_pos_indices.append(i)

print(f"Found {len(val_pos_indices)} validation images containing class {TARGET_ID} (out of {len(val_set)} total).")

# Create Subset
val_subset = Subset(val_set, val_pos_indices)

# Create Loader
val_loader_pos = DataLoader(
    val_subset,
    batch_size=2,
    sampler=SequentialSampler(val_subset),
    num_workers=2,
    pin_memory=True,
)

Scanning validation dataset for images containing class 28...
Found 75 validation images containing class 28 (out of 500 total).


In [ ]:
from torch.cuda.amp import GradScaler, autocast

# 1. Initialize Mixed Precision Scaler
scaler = GradScaler(enabled=torch.cuda.is_available())

model.train()
steps = 10000  # Approx 100 epochs
print_freq = 50
running_loss = 0.0

# Create an infinite iterator for the loop
it = iter(train_loader_pos)

print(f"Starting unweighted training for {steps} steps...")

for step in range(1, steps + 1):
    # 2. Infinite Loader Logic
    try:
        imgs, lbls = next(it)
    except StopIteration:
        it = iter(train_loader_pos)
        imgs, lbls = next(it)

    # 3. Move to GPU
    imgs = imgs.to(device, non_blocking=True)
    target_bin = lbls.to(device, non_blocking=True)

    # 4. Forward Pass (Mixed Precision)
    with autocast(enabled=torch.cuda.is_available()):
        out = model.forward_with_new(imgs)
        logits_new = out["logits_new_2ch"]

        # This uses the 'crit' you defined earlier (Plain CE)
        loss = crit(logits_new, target_bin)

    # 5. Backward Pass
    optim.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optim)
    scaler.update()

    # 6. Logging
    running_loss += loss.item()
    if step % print_freq == 0:
        avg_loss = running_loss / print_freq
        print(f"[Step {step}/{steps}] Loss: {avg_loss:.4f}")
        running_loss = 0.0

print("Training finished.")

/tmp/ipython-input-4158620584.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())


Starting unweighted training for 10000 steps...


/tmp/ipython-input-4158620584.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


[Step 50/10000] Loss: 0.5347
[Step 100/10000] Loss: 0.0877
[Step 150/10000] Loss: 0.0696
[Step 200/10000] Loss: 0.0670
[Step 250/10000] Loss: 0.0603
[Step 300/10000] Loss: 0.0404
[Step 350/10000] Loss: 0.0412
[Step 400/10000] Loss: 0.0365
[Step 450/10000] Loss: 0.0394
[Step 500/10000] Loss: 0.0336
[Step 550/10000] Loss: 0.0266
[Step 600/10000] Loss: 0.0245
[Step 650/10000] Loss: 0.0327
[Step 700/10000] Loss: 0.0388
[Step 750/10000] Loss: 0.0233
[Step 800/10000] Loss: 0.0233
[Step 850/10000] Loss: 0.0207
[Step 900/10000] Loss: 0.0225
[Step 950/10000] Loss: 0.0228
[Step 1000/10000] Loss: 0.0193
[Step 1050/10000] Loss: 0.0217
[Step 1100/10000] Loss: 0.0198
[Step 1150/10000] Loss: 0.0140
[Step 1200/10000] Loss: 0.0248
[Step 1250/10000] Loss: 0.0190
[Step 1300/10000] Loss: 0.0196
[Step 1350/10000] Loss: 0.0199
[Step 1400/10000] Loss: 0.0183
[Step 1450/10000] Loss: 0.0149
[Step 1500/10000] Loss: 0.0163
[Step 1550/10000] Loss: 0.0151
[Step 1600/10000] Loss: 0.0115
[Step 1650/10000] Loss: 0.01

Evaluate

In [ ]:
import torch

@torch.no_grad()
def eval_new_head_at_tau(model, loader, tau=0.50):
    model.eval()
    tp = fp = fn = 0
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        tgt  = lbls.to(device)                 # {0,1,255}
        out  = model.forward_with_new(imgs)
        logits = out["logits_new_2ch"]         # [B,2,H,W]
        # get probability of "new" via softmax
        p_new = torch.softmax(logits, dim=1)[:, 1]   # [B,H,W]
        pred = (p_new >= tau).long()                 # thresholded prediction {0,1}
        valid = tgt != 255
        tp += ((pred==1) & (tgt==1) & valid).sum().item()
        fp += ((pred==1) & (tgt==0) & valid).sum().item()
        fn += ((pred==0) & (tgt==1) & valid).sum().item()
    iou = tp / max(1, tp+fp+fn)
    prec = tp / max(1, tp+fp)
    rec  = tp / max(1, tp+fn)
    f1   = 2*prec*rec / max(1e-9, prec+rec)
    return {"tau": tau, "IoU": iou, "precision": prec, "recall": rec, "F1": f1, "tp": tp, "fp": fp, "fn": fn}

In [ ]:
taus = np.round(np.arange(0.05, 0.96, 0.05), 2)   # 0.05 → 0.95 in 0.05 steps
rows = [eval_new_head_at_tau(model, val_loader_pos, tau=t) for t in taus]
rows_sorted = sorted(rows, key=lambda x: x["IoU"], reverse=True)

print("Top by IoU:")
for r in rows_sorted[:5]:
    print(f"τ={r['tau']:.2f}  IoU={r['IoU']:.4f}  P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['F1']:.3f}  tp={r['tp']} fp={r['fp']} fn={r['fn']}")

Top by IoU:
τ=0.40  IoU=0.7049  P=0.844  R=0.810  F1=0.827  tp=2886451 fp=532005 fn=676669
τ=0.45  IoU=0.7047  P=0.852  R=0.803  F1=0.827  tp=2860712 fp=496587 fn=702408
τ=0.35  IoU=0.7044  P=0.836  R=0.817  F1=0.827  tp=2912226 fp=571046 fn=650894
τ=0.50  IoU=0.7040  P=0.859  R=0.796  F1=0.826  tp=2835321 fp=464194 fn=727799
τ=0.30  IoU=0.7032  P=0.827  R=0.825  F1=0.826  tp=2938598 fp=615766 fn=624522


In [ ]:
import torch

TAU = 0.40

@torch.no_grad()
def fuse_hard_gate(out_15, out_new2, tau=TAU):
    """
    out_15: [B,15,H,W] logits from the old head
    out_new2: [B,2,H,W] logits from the new head (bg,new)
    returns: fused logits [B,16,H,W] and fused prediction [B,H,W] in {0..15}
    """
    # probability of new class
    p_new = torch.softmax(out_new2, dim=1)[:, 1:2]  # [B,1,H,W]
    new_mask = (p_new >= tau)                        # bool mask

    # base prediction among first 15 classes
    pred15 = out_15.argmax(dim=1, keepdim=True)      # [B,1,H,W]

    # fused prediction: class 16 where mask true, else pred15
    fused_pred = pred15.squeeze(1).clone()           # [B,H,W], values 0..14
    fused_pred[new_mask.squeeze(1)] = 15             # use 15 as index of the new class

    # if you also want fused logits (for saving / downstream), build a 16-ch tensor
    B, _, H, W = out_15.shape
    fused_logits = torch.zeros(B, 16, H, W, device=out_15.device, dtype=out_15.dtype)
    fused_logits[:, :15] = out_15
    # put new head logit for the "new" channel at index 15
    fused_logits[:, 15:16] = out_new2[:, 1:2]

    return fused_logits, fused_pred

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import glob
from tqdm import tqdm
import torch.nn.functional as F

# --- 1. Define the 16-Class Dataset ---
class CityscapesFull16(Dataset):
    def __init__(self, root, split="val", new_class_orig_id=28):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir  = os.path.join(root, "gtFine", split)

        # 1. Define Mapping: Raw ID (0-33) -> Model ID (0-15)
        # We start with the ignore label (255) for everything
        self.id_map = {i: 255 for i in range(35)}

        # Map the original 15 classes (from your cityscapes.py)
        # 7:0 (Road), 8:1 (Sidewalk), etc.
        base_mapping = {
            7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5,
            19: 6, 20: 7, 21: 8, 22: 9, 23: 10, 24: 11,
            25: 12, 26: 13, 27: 14
        }
        self.id_map.update(base_mapping)

        # Map the NEW Class (Bus 28 -> 15)
        self.id_map[new_class_orig_id] = 15

        self.items = []
        # Find all images
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                city_img_dir = os.path.join(self.img_dir, city)
                # glob might return unsorted, so we sort to be safe
                imgs = sorted(glob.glob(os.path.join(city_img_dir, "*_leftImg8bit.png")))
                for ip in imgs:
                    base_name = os.path.basename(ip).replace("_leftImg8bit.png", "")
                    gt_name = base_name + "_gtFine_labelIds.png"
                    gp = os.path.join(self.gt_dir, city, gt_name)
                    if os.path.exists(gp):
                        self.items.append((ip, gp))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ip, gp = self.items[idx]

        # Load Image (RGB) - Matches your Base Model training
        img = Image.open(ip).convert("RGB")
        img_t = img_tf(img)

        # Load Label (Raw IDs)
        lbl_raw = np.array(Image.open(gp), dtype=np.uint8)

        # Remap to 0-15
        lbl_16 = np.full_like(lbl_raw, 255)
        for raw_id, train_id in self.id_map.items():
            lbl_16[lbl_raw == raw_id] = train_id

        lbl_t = torch.from_numpy(lbl_16).long()
        return img_t, lbl_t

# --- 2. Setup Data ---
val_set_16 = CityscapesFull16(DATA_ROOT, split="val")
val_loader_16 = DataLoader(val_set_16, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# Class Names
CLASS_NAMES = [
    "Road", "Sidewalk", "Building", "Wall", "Fence",
    "Pole", "Traffic Light", "Traffic Sign", "Vegetation", "Terrain",
    "Sky", "Person", "Rider", "Car", "Truck",
    "BUS (New)"
]

# --- 3. Evaluation Loop ---
def evaluate_all_classes(model, loader, device, tau=0.55):
    model.eval()

    # Confusion Matrix: [16, 16] (Rows: GT, Cols: Pred)
    num_classes = 16
    confusion_matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)

    print(f"Evaluating 16-class mIoU on {len(loader.dataset)} images with tau={tau}...")

    with torch.no_grad():
        for imgs, targets in tqdm(loader):
            imgs = imgs.to(device)
            targets = targets.to(device)

            # Forward
            out = model.forward_with_new(imgs)

            # Upsample Logits to Target Size (CRITICAL for accuracy)
            h, w = targets.shape[-2:]
            logits_15 = F.interpolate(out["logits_15"], size=(h,w), mode='bilinear', align_corners=True)
            logits_new = F.interpolate(out["logits_new_2ch"], size=(h,w), mode='bilinear', align_corners=True)

            # Fuse
            _, preds = fuse_hard_gate(logits_15, logits_new, tau=tau)

            # Update Matrix (Vectorized)
            mask = (targets != 255)
            t_flat = targets[mask]
            p_flat = preds[mask]

            # Add to confusion matrix
            # Index = GT * num_classes + Pred
            if t_flat.numel() > 0:
                indices = t_flat * num_classes + p_flat
                cnts = torch.bincount(indices, minlength=num_classes**2)
                confusion_matrix += cnts.view(num_classes, num_classes)

    # --- 4. Calculate IoU ---
    tp = torch.diag(confusion_matrix)
    row_sum = confusion_matrix.sum(dim=1) # GT count
    col_sum = confusion_matrix.sum(dim=0) # Pred count
    union = row_sum + col_sum - tp

    iou = tp.float() / (union.float() + 1e-6)

    # Print Results
    print("\n" + "="*55)
    print(f"{'ID':<3} | {'Class Name':<15} | {'IoU (%)':<10}")
    print("-" * 55)

    iou_list = []
    for i, score in enumerate(iou):
        val = score.item() * 100
        print(f"{i:<3} | {CLASS_NAMES[i]:<15} | {val:0.2f}")
        iou_list.append(val)

    print("-" * 55)
    # Mean IoU (Original 15)
    miou_15 = np.mean(iou_list[:15])
    # Mean IoU (All 16)
    miou_16 = np.mean(iou_list)

    print(f"mIoU (Original 15 Classes): {miou_15:.2f}%")
    print(f"mIoU (Total 16 Classes):    {miou_16:.2f}%")
    print("="*55)

# --- Run It ---
evaluate_all_classes(model, val_loader_16, device, tau=0.55)

Evaluating 16-class mIoU on 500 images with tau=0.55...


100%|██████████| 125/125 [02:33<00:00,  1.23s/it]


ID  | Class Name      | IoU (%)   
-------------------------------------------------------
0   | Road            | 80.33
1   | Sidewalk        | 79.99
2   | Building        | 80.23
3   | Wall            | 33.66
4   | Fence           | 42.65
5   | Pole            | 43.37
6   | Traffic Light   | 40.56
7   | Traffic Sign    | 53.01
8   | Vegetation      | 64.41
9   | Terrain         | 52.63
10  | Sky             | 88.61
11  | Person          | 63.47
12  | Rider           | 39.15
13  | Car             | 85.13
14  | Truck           | 18.43
15  | BUS (New)       | 37.21
-------------------------------------------------------
mIoU (Original 15 Classes): 57.71%
mIoU (Total 16 Classes):    56.43%


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import glob
from tqdm import tqdm
import torch.nn.functional as F
import torchvision.transforms as T

# --- 1. Define BGR Transform ---
# This matches the preprocessing used by the original PIDNet backbone (cv2.imread)
# We apply the standard ImageNet mean/std to the BGR image.
norm_mean = [0.485, 0.456, 0.406]
norm_std  = [0.229, 0.224, 0.225]

def transform_bgr(pil_img):
    # Convert PIL (RGB) to Numpy
    img_np = np.array(pil_img)
    # Convert RGB -> BGR
    img_bgr = img_np[:, :, ::-1].copy()
    # To Tensor (C, H, W) and Normalize
    tensor = torch.from_numpy(img_bgr).float().permute(2, 0, 1) / 255.0
    tensor = T.Normalize(mean=norm_mean, std=norm_std)(tensor)
    return tensor

# --- 2. Define the 16-Class Dataset ---
class CityscapesFull16(Dataset):
    def __init__(self, root, split="val", new_class_orig_id=28):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir  = os.path.join(root, "gtFine", split)

        # Mapping: Raw ID (0-33) -> Model ID (0-15)
        self.id_map = {i: 255 for i in range(35)}

        # Original 15 classes
        base_mapping = {
            7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5,
            19: 6, 20: 7, 21: 8, 22: 9, 23: 10, 24: 11,
            25: 12, 26: 13, 27: 14
        }
        self.id_map.update(base_mapping)

        # New Class (Bus)
        self.id_map[new_class_orig_id] = 15

        self.items = []
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                city_img_dir = os.path.join(self.img_dir, city)
                imgs = sorted(glob.glob(os.path.join(city_img_dir, "*_leftImg8bit.png")))
                for ip in imgs:
                    base_name = os.path.basename(ip).replace("_leftImg8bit.png", "")
                    gt_name = base_name + "_gtFine_labelIds.png"
                    gp = os.path.join(self.gt_dir, city, gt_name)
                    if os.path.exists(gp):
                        self.items.append((ip, gp))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ip, gp = self.items[idx]

        # Load Image (RGB)
        img = Image.open(ip).convert("RGB")

        # FIX: Apply BGR Transform
        img_t = transform_bgr(img)

        # Load Label
        lbl_raw = np.array(Image.open(gp), dtype=np.uint8)
        lbl_16 = np.full_like(lbl_raw, 255)
        for raw_id, train_id in self.id_map.items():
            lbl_16[lbl_raw == raw_id] = train_id

        lbl_t = torch.from_numpy(lbl_16).long()
        return img_t, lbl_t

# --- 3. Setup Data ---
val_set_16 = CityscapesFull16(DATA_ROOT, split="val")
val_loader_16 = DataLoader(val_set_16, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# Class Names
CLASS_NAMES = [
    "Road", "Sidewalk", "Building", "Wall", "Fence",
    "Pole", "Traffic Light", "Traffic Sign", "Vegetation", "Terrain",
    "Sky", "Person", "Rider", "Car", "Truck",
    "BUS (New)"
]

# --- 4. Helper Function: Fuse Hard Gate (Ensure it's defined) ---
@torch.no_grad()
def fuse_hard_gate(out_15, out_new2, tau):
    p_new = torch.softmax(out_new2, dim=1)[:, 1:2]
    new_mask = (p_new >= tau)
    pred15 = out_15.argmax(dim=1, keepdim=True)
    fused_pred = pred15.squeeze(1).clone()
    fused_pred[new_mask.squeeze(1)] = 15

    # Optional: Construct fused logits if needed (not used for simple IoU calc)
    # fused_logits = ...
    return None, fused_pred

# --- 5. Evaluation Loop ---
def evaluate_all_classes(model, loader, device, tau=0.55):
    model.eval()

    num_classes = 16
    confusion_matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)

    print(f"Evaluating 16-class mIoU on {len(loader.dataset)} images (BGR Input) with tau={tau}...")

    with torch.no_grad():
        for imgs, targets in tqdm(loader):
            imgs = imgs.to(device)
            targets = targets.to(device)

            # Forward
            out = model.forward_with_new(imgs)

            # Upsample (CRITICAL)
            h, w = targets.shape[-2:]
            logits_15 = F.interpolate(out["logits_15"], size=(h,w), mode='bilinear', align_corners=True)
            logits_new = F.interpolate(out["logits_new_2ch"], size=(h,w), mode='bilinear', align_corners=True)

            # Fuse
            _, preds = fuse_hard_gate(logits_15, logits_new, tau=tau)

            # Update Matrix
            mask = (targets != 255)
            t_flat = targets[mask]
            p_flat = preds[mask]

            if t_flat.numel() > 0:
                indices = t_flat * num_classes + p_flat
                cnts = torch.bincount(indices, minlength=num_classes**2)
                confusion_matrix += cnts.view(num_classes, num_classes)

    # Calculate IoU
    tp = torch.diag(confusion_matrix)
    row_sum = confusion_matrix.sum(dim=1)
    col_sum = confusion_matrix.sum(dim=0)
    union = row_sum + col_sum - tp
    iou = tp.float() / (union.float() + 1e-6)

    # Print
    print("\n" + "="*55)
    print(f"{'ID':<3} | {'Class Name':<15} | {'IoU (%)':<10}")
    print("-" * 55)

    iou_list = []
    for i, score in enumerate(iou):
        val = score.item() * 100
        print(f"{i:<3} | {CLASS_NAMES[i]:<15} | {val:0.2f}")
        iou_list.append(val)

    print("-" * 55)
    miou_15 = np.mean(iou_list[:15])
    miou_16 = np.mean(iou_list)

    print(f"mIoU (Original 15 Classes): {miou_15:.2f}%")
    print(f"mIoU (Total 16 Classes):    {miou_16:.2f}%")
    print("="*55)

# --- Run It ---
evaluate_all_classes(model, val_loader_16, device, tau=0.55)

Evaluating 16-class mIoU on 500 images (BGR Input) with tau=0.55...


100%|██████████| 125/125 [00:52<00:00,  2.36it/s]


ID  | Class Name      | IoU (%)   
-------------------------------------------------------
0   | Road            | 71.33
1   | Sidewalk        | 66.96
2   | Building        | 77.10
3   | Wall            | 18.86
4   | Fence           | 38.54
5   | Pole            | 37.44
6   | Traffic Light   | 36.45
7   | Traffic Sign    | 43.24
8   | Vegetation      | 59.49
9   | Terrain         | 31.70
10  | Sky             | 68.61
11  | Person          | 59.63
12  | Rider           | 36.21
13  | Car             | 74.69
14  | Truck           | 17.15
15  | BUS (New)       | 34.78
-------------------------------------------------------
mIoU (Original 15 Classes): 49.16%
mIoU (Total 16 Classes):    48.26%


In [ ]:
# Copy/Paste this back to define the Dataset as RGB again
class CityscapesFull16(Dataset):
    def __init__(self, root, split="val", new_class_orig_id=28):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir  = os.path.join(root, "gtFine", split)
        self.id_map = {i: 255 for i in range(35)}
        base_mapping = {
            7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5,
            19: 6, 20: 7, 21: 8, 22: 9, 23: 10, 24: 11,
            25: 12, 26: 13, 27: 14
        }
        self.id_map.update(base_mapping)
        self.id_map[new_class_orig_id] = 15
        self.items = []
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                city_img_dir = os.path.join(self.img_dir, city)
                imgs = sorted(glob.glob(os.path.join(city_img_dir, "*_leftImg8bit.png")))
                for ip in imgs:
                    base_name = os.path.basename(ip).replace("_leftImg8bit.png", "")
                    gt_name = base_name + "_gtFine_labelIds.png"
                    gp = os.path.join(self.gt_dir, city, gt_name)
                    if os.path.exists(gp):
                        self.items.append((ip, gp))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ip, gp = self.items[idx]

        # Load Image (RGB)
        img = Image.open(ip).convert("RGB")
        img_t = img_tf(img) # Use your standard RGB transform

        # Load Label
        lbl_raw = np.array(Image.open(gp), dtype=np.uint8)
        lbl_16 = np.full_like(lbl_raw, 255)
        for raw_id, train_id in self.id_map.items():
            lbl_16[lbl_raw == raw_id] = train_id

        lbl_t = torch.from_numpy(lbl_16).long()
        return img_t, lbl_t

# Re-create Loader
val_set_16 = CityscapesFull16(DATA_ROOT, split="val")
val_loader_16 = DataLoader(val_set_16, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print("RGB Dataset Restored.")

RGB Dataset Restored.


In [ ]:
# Evaluate with tau=1.0 to disable the new branch
print("Running DIAGNOSTIC evaluation (Base Model Only)...")
evaluate_all_classes(model, val_loader_16, device, tau=1.0)

Running DIAGNOSTIC evaluation (Base Model Only)...
Evaluating 16-class mIoU on 500 images (BGR Input) with tau=1.0...


100%|██████████| 125/125 [00:44<00:00,  2.83it/s]


ID  | Class Name      | IoU (%)   
-------------------------------------------------------
0   | Road            | 80.35
1   | Sidewalk        | 79.98
2   | Building        | 80.36
3   | Wall            | 33.21
4   | Fence           | 42.92
5   | Pole            | 43.10
6   | Traffic Light   | 40.02
7   | Traffic Sign    | 51.75
8   | Vegetation      | 64.26
9   | Terrain         | 52.59
10  | Sky             | 88.55
11  | Person          | 63.39
12  | Rider           | 39.12
13  | Car             | 85.51
14  | Truck           | 29.33
15  | BUS (New)       | 2.51
-------------------------------------------------------
mIoU (Original 15 Classes): 58.30%
mIoU (Total 16 Classes):    54.81%


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import glob
from tqdm import tqdm
import torch.nn.functional as F
import torchvision.transforms as T

# Standard ImageNet normalization (RGB)
img_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

# --- RGB Dataset Class (Restored) ---
class CityscapesFull16(Dataset):
    def __init__(self, root, split="val", new_class_orig_id=28):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir  = os.path.join(root, "gtFine", split)
        self.id_map = {i: 255 for i in range(35)}

        # Original 15 classes
        base_mapping = {
            7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5,
            19: 6, 20: 7, 21: 8, 22: 9, 23: 10, 24: 11,
            25: 12, 26: 13, 27: 14
        }
        self.id_map.update(base_mapping)
        self.id_map[new_class_orig_id] = 15 # Bus

        self.items = []
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                city_img_dir = os.path.join(self.img_dir, city)
                imgs = sorted(glob.glob(os.path.join(city_img_dir, "*_leftImg8bit.png")))
                for ip in imgs:
                    base_name = os.path.basename(ip).replace("_leftImg8bit.png", "")
                    gt_name = base_name + "_gtFine_labelIds.png"
                    gp = os.path.join(self.gt_dir, city, gt_name)
                    if os.path.exists(gp):
                        self.items.append((ip, gp))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ip, gp = self.items[idx]

        # Load Image (RGB) - This gave the best result (58%)
        img = Image.open(ip).convert("RGB")
        img_t = img_tf(img)

        # Load Label
        lbl_raw = np.array(Image.open(gp), dtype=np.uint8)
        lbl_16 = np.full_like(lbl_raw, 255)
        for raw_id, train_id in self.id_map.items():
            lbl_16[lbl_raw == raw_id] = train_id

        lbl_t = torch.from_numpy(lbl_16).long()
        return img_t, lbl_t

# Setup Data
val_set_16 = CityscapesFull16(DATA_ROOT, split="val")
val_loader_16 = DataLoader(val_set_16, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# Run Final Evaluation with your best Tau
print("Restored RGB Dataset. Running final evaluation with Tau=0.40...")
evaluate_all_classes(model, val_loader_16, device, tau=0.40)

Restored RGB Dataset. Running final evaluation with Tau=0.40...
Evaluating 16-class mIoU on 500 images (BGR Input) with tau=0.4...


100%|██████████| 125/125 [00:44<00:00,  2.83it/s]


ID  | Class Name      | IoU (%)   
-------------------------------------------------------
0   | Road            | 80.31
1   | Sidewalk        | 79.99
2   | Building        | 80.30
3   | Wall            | 33.36
4   | Fence           | 42.66
5   | Pole            | 43.36
6   | Traffic Light   | 40.56
7   | Traffic Sign    | 52.94
8   | Vegetation      | 64.39
9   | Terrain         | 52.60
10  | Sky             | 88.59
11  | Person          | 63.48
12  | Rider           | 39.14
13  | Car             | 85.26
14  | Truck           | 19.89
15  | BUS (New)       | 37.88
-------------------------------------------------------
mIoU (Original 15 Classes): 57.79%
mIoU (Total 16 Classes):    56.54%


In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F

# (Assuming CityscapesFull16 and transform_bgr are already defined from previous cells)
# If not, ensure you run the cell defining the 'RGB' dataset class first.

def evaluate_dual_mode(model, loader, device, tau=0.55):
    model.eval()

    # We need two matrices to show you the difference
    # 1. Honest Matrix (16x16): Grades everything.
    cm_honest = torch.zeros((16, 16), dtype=torch.int64, device=device)

    # 2. Legacy Matrix (15x15): Ignores Bus pixels (mimics old training).
    cm_legacy = torch.zeros((15, 15), dtype=torch.int64, device=device)

    print(f"Evaluating Dual-Mode mIoU (Tau={tau})...")

    with torch.no_grad():
        for imgs, targets in tqdm(loader):
            imgs = imgs.to(device)
            targets = targets.to(device)

            # Forward
            out = model.forward_with_new(imgs)

            # Upsample
            h, w = targets.shape[-2:]
            logits_15 = F.interpolate(out["logits_15"], size=(h,w), mode='bilinear', align_corners=True)
            logits_new = F.interpolate(out["logits_new_2ch"], size=(h,w), mode='bilinear', align_corners=True)

            # Fuse
            _, preds = fuse_hard_gate(logits_15, logits_new, tau=tau)

            # --- UPDATE HONEST MATRIX (16 Classes) ---
            mask_16 = (targets != 255)
            if mask_16.any():
                t16 = targets[mask_16]
                p16 = preds[mask_16]
                indices = t16 * 16 + p16
                cm_honest += torch.bincount(indices, minlength=16**2).view(16, 16)

            # --- UPDATE LEGACY MATRIX (15 Classes) ---
            # mimic old behavior: Ignore pixels where GT is Bus (15)
            mask_15 = (targets != 255) & (targets != 15)

            if mask_15.any():
                t15 = targets[mask_15]
                p15 = preds[mask_15]

                # CRITICAL: If the model predicted Bus (15), we must clip it to
                # something valid or just mask those out too?
                # Actually, in the old model, 15 was impossible.
                # Here, if model predicts 15, we treat it as a "wrong" prediction for 0-14.
                # However, to map 16x16 preds to 15x15 matrix, we simply clamp or ignore.
                # Let's count predictions of "Bus" as "False Positive" for nobody
                # (since it's not 0-14). We filter prediction 15 out of the view.

                valid_preds = (p15 < 15)
                if valid_preds.any():
                    t15 = t15[valid_preds]
                    p15 = p15[valid_preds]
                    indices_15 = t15 * 15 + p15
                    cm_legacy += torch.bincount(indices_15, minlength=15**2).view(15, 15)

    # --- Print Results ---

    # 1. Honest Score
    tp = torch.diag(cm_honest)
    union = cm_honest.sum(1) + cm_honest.sum(0) - tp
    iou_16 = tp.float() / (union.float() + 1e-6)
    miou_16 = iou_16.mean().item() * 100

    # 2. Legacy Score
    tp_leg = torch.diag(cm_legacy)
    union_leg = cm_legacy.sum(1) + cm_legacy.sum(0) - tp_leg
    iou_leg = tp_leg.float() / (union_leg.float() + 1e-6)
    miou_15_legacy = iou_leg.mean().item() * 100

    print("\n" + "="*60)
    print(f"{'Class Name':<15} | {'Honest (16)':<12} | {'Legacy (15-view)':<12}")
    print("-" * 60)

    for i in range(15):
        print(f"{CLASS_NAMES[i]:<15} | {iou_16[i]*100:0.2f}%        | {iou_leg[i]*100:0.2f}%")

    print(f"{CLASS_NAMES[15]:<15} | {iou_16[15]*100:0.2f}%        | --")
    print("-" * 60)
    print(f"Mean IoU       | {miou_16:0.2f}%        | {miou_15_legacy:0.2f}%")
    print("="*60)
    print("NOTE: 'Legacy' column ignores all Bus pixels, matching original training.")

# Run it
evaluate_dual_mode(model, val_loader_16, device, tau=0.55)

Evaluating Dual-Mode mIoU (Tau=0.55)...


100%|██████████| 125/125 [00:43<00:00,  2.88it/s]


Class Name      | Honest (16)  | Legacy (15-view)
------------------------------------------------------------
Road            | 80.32%        | 80.36%
Sidewalk        | 79.99%        | 80.02%
Building        | 80.33%        | 80.50%
Wall            | 33.37%        | 33.65%
Fence           | 42.72%        | 43.34%
Pole            | 43.35%        | 43.49%
Traffic Light   | 40.54%        | 40.67%
Traffic Sign    | 52.91%        | 53.45%
Vegetation      | 64.38%        | 64.43%
Terrain         | 52.60%        | 52.62%
Sky             | 88.59%        | 88.59%
Person          | 63.50%        | 63.65%
Rider           | 39.15%        | 39.25%
Car             | 85.54%        | 88.55%
Truck           | 20.69%        | 28.17%
BUS (New)       | 39.92%        | --
------------------------------------------------------------
Mean IoU       | 56.74%        | 58.71%
NOTE: 'Legacy' column ignores all Bus pixels, matching original training.


In [ ]:
# Test with a strict threshold to protect the Old Classes
STRICT_TAU = 0.85

print(f"--- Diagnosing Truck Overwriting (Tau={STRICT_TAU}) ---")
evaluate_dual_mode(model, val_loader_16, device, tau=STRICT_TAU)

--- Diagnosing Truck Overwriting (Tau=0.85) ---
Evaluating Dual-Mode mIoU (Tau=0.85)...


100%|██████████| 125/125 [00:46<00:00,  2.69it/s]


Class Name      | Honest (16)  | Legacy (15-view)
------------------------------------------------------------
Road            | 80.34%        | 80.36%
Sidewalk        | 79.99%        | 80.01%
Building        | 80.37%        | 80.48%
Wall            | 33.36%        | 33.62%
Fence           | 42.84%        | 43.30%
Pole            | 43.32%        | 43.46%
Traffic Light   | 40.49%        | 40.63%
Traffic Sign    | 52.81%        | 53.32%
Vegetation      | 64.36%        | 64.42%
Terrain         | 52.59%        | 52.61%
Sky             | 88.57%        | 88.58%
Person          | 63.52%        | 63.62%
Rider           | 39.17%        | 39.25%
Car             | 86.11%        | 88.43%
Truck           | 22.64%        | 28.52%
BUS (New)       | 44.03%        | --
------------------------------------------------------------
Mean IoU       | 57.16%        | 58.71%
NOTE: 'Legacy' column ignores all Bus pixels, matching original training.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import GradScaler, autocast
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
from models.pidnet_cl import PIDNetCL

# ==========================================
# 1. SETUP: RGB DATASET (Official Config)
# ==========================================
DATA_ROOT = "/content/drive/MyDrive/CSS/PIDNet/data/cityscapes"
NEW_CLASS_ID = 28
img_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # RGB Stats
])

class CityscapesCL(Dataset):
    def __init__(self, root, split="train"):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir = os.path.join(root, "gtFine", split)
        self.items = []
        # Find images
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                imgs = sorted(glob.glob(os.path.join(self.img_dir, city, "*_leftImg8bit.png")))
                for ip in imgs:
                    gp = os.path.join(self.gt_dir, city, os.path.basename(ip).replace("_leftImg8bit.png", "_gtFine_labelIds.png"))
                    if os.path.exists(gp): self.items.append((ip, gp))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        ip, gp = self.items[idx]
        # RGB Loading (Matches Official PIDNet training)
        img = img_tf(Image.open(ip).convert("RGB"))
        lbl = np.array(Image.open(gp), dtype=np.uint8)

        # Binary Target for Training (Bus vs Rest)
        tgt = np.full_like(lbl, 255)
        valid_bg = [7,8,11,12,13,17,19,20,21,22,23,24,25,26,27]
        for c in valid_bg: tgt[lbl == c] = 0
        tgt[lbl == NEW_CLASS_ID] = 1
        return img, torch.from_numpy(tgt).long(), lbl # Return raw label for eval

# ==========================================
# 2. MODEL: LOAD & FREEZE
# ==========================================
print("=> Initializing Model & Loading Base Weights...")
model = PIDNetCL(m=2, n=3, num_classes=15, planes=32, ppm_planes=96, head_planes=128, augment=True)
ckpt = torch.load("/content/drive/MyDrive/CSS/PIDNet/output/cityscapes/pidnet_s_custom_15/best.pt", map_location="cpu")
state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
state_dict = {k.replace('module.', '').replace('model.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)

# Freeze Old, Thaw New
for name, p in model.named_parameters():
    p.requires_grad = any(t in name for t in ["layer3_d_new","layer4_d_new","diff3_new","diff4_new","seghead_new"])
model.cuda()
model.train()

# ==========================================
# 3. TRAIN: SUBSET ONLY
# ==========================================
print("=> Preparing Training Data...")
train_set = CityscapesCL(DATA_ROOT, "train")
bus_indices = [i for i, (_, gp) in enumerate(train_set.items) if (np.array(Image.open(gp)) == NEW_CLASS_ID).any()]
train_loader = DataLoader(Subset(train_set, bus_indices), batch_size=6, shuffle=True, num_workers=2, pin_memory=True)

print(f"=> Retraining New Branch (RGB) on {len(bus_indices)} images...")
optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
crit = nn.CrossEntropyLoss(ignore_index=255)
scaler = GradScaler(enabled=True)

for epoch in range(5): # Fast retraining (5 epochs of the subset is enough to recover)
    for i, (img, tgt, _) in enumerate(train_loader):
        img, tgt = img.cuda(), tgt.cuda()
        with autocast(enabled=True):
            out = model.forward_with_new(img)
            loss = crit(out["logits_new_2ch"], tgt)
        optim.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()
    print(f"   Epoch {epoch+1} Loss: {loss.item():.4f}")

# ==========================================
# 4. EVALUATE: DUAL MODE (LEGACY vs HONEST)
# ==========================================
print("\n=> Running Dual-Mode Evaluation (Validation Set)...")
val_set = CityscapesCL(DATA_ROOT, "val")
val_loader = DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)

@torch.no_grad()
def fuse(o15, onew, tau=0.55):
    p_new = torch.softmax(onew, dim=1)[:, 1:2]
    return torch.where(p_new >= tau, torch.tensor(15).to(p_new.device), o15.argmax(1, keepdim=True)).squeeze(1)

model.eval()
# Confusion Matrices
cm_legacy = torch.zeros((15, 15), dtype=torch.long).cuda() # Original 15 classes
cm_honest = torch.zeros((16, 16), dtype=torch.long).cuda() # All 16 classes

id_map_15 = {7:0,8:1,11:2,12:3,13:4,17:5,19:6,20:7,21:8,22:9,23:10,24:11,25:12,26:13,27:14}

for img, _, raw_lbl in tqdm(val_loader):
    img = img.cuda()
    raw_lbl = raw_lbl.long().cuda()

    # 1. Forward & Up
    out = model.forward_with_new(img)
    l15 = F.interpolate(out["logits_15"], raw_lbl.shape[-2:], mode='bilinear', align_corners=True)
    lnew = F.interpolate(out["logits_new_2ch"], raw_lbl.shape[-2:], mode='bilinear', align_corners=True)
    pred = fuse(l15, lnew, tau=0.55) # Use 0.55 or adjust higher if Truck drops too much

    # 2. Prepare Targets (16 Class)
    tgt_16 = torch.full_like(raw_lbl, 255)
    for k, v in id_map_15.items(): tgt_16[raw_lbl == k] = v
    tgt_16[raw_lbl == NEW_CLASS_ID] = 15

    # 3. Prepare Targets (15 Class Legacy - Ignore Bus)
    tgt_15 = tgt_16.clone()
    tgt_15[tgt_15 == 15] = 255 # IGNORE BUS PIXELS (Matches Base Model Evaluation)

    # 4. Update Matrices
    mask16 = tgt_16 != 255
    if mask16.any():
        cm_honest += torch.bincount(tgt_16[mask16]*16 + pred[mask16], minlength=256).view(16,16)

    mask15 = tgt_15 != 255
    if mask15.any():
        # Map predictions of "15" (Bus) back to something else or just mask them?
        # In legacy mode, if we predict "Bus", it's technically a "Non-15-class" prediction.
        # But usually we just calculate IoU on 0-14.
        p15_view = pred[mask15]
        # Only count if prediction is 0-14 (Valid Legacy)
        valid_p = p15_view < 15
        if valid_p.any():
            cm_legacy += torch.bincount(tgt_15[mask15][valid_p]*15 + p15_view[valid_p], minlength=225).view(15,15)

# Calculate Scores
iou_leg = torch.diag(cm_legacy) / (cm_legacy.sum(1) + cm_legacy.sum(0) - torch.diag(cm_legacy) + 1e-6)
iou_hon = torch.diag(cm_honest) / (cm_honest.sum(1) + cm_honest.sum(0) - torch.diag(cm_honest) + 1e-6)

print(f"\nLegacy mIoU (15 Class): {iou_leg.mean().item()*100:.2f}%  -- Should be around 79%")
print(f"Honest mIoU (16 Class): {iou_hon.mean().item()*100:.2f}%  -- Real System Performance")
print(f"Truck (Legacy): {iou_leg[14].item()*100:.2f}% | Bus (Honest): {iou_hon[15].item()*100:.2f}%")

=> Initializing Model & Loading Base Weights...
=> Preparing Training Data...
=> Retraining New Branch (RGB) on 274 images...


/tmp/ipython-input-1119045269.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=True)
/tmp/ipython-input-1119045269.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):


   Epoch 1 Loss: 0.0747
   Epoch 2 Loss: 0.0430
   Epoch 3 Loss: 0.0340
   Epoch 4 Loss: 0.0970
   Epoch 5 Loss: 0.0253

=> Running Dual-Mode Evaluation (Validation Set)...


100%|██████████| 125/125 [00:42<00:00,  2.93it/s]


Legacy mIoU (15 Class): 77.22%  <-- Should be ~79%
Honest mIoU (16 Class): 72.65%  <-- Real System Performance
Truck (Legacy): 76.24% | Bus (Honest): 30.93%


P-branch

In [ ]:
!cp -f /content/drive/MyDrive/CSS/model/pidnet_cl_p.py /content/PIDNet/models/pidnet_cl_p.py

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import GradScaler, autocast
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import numpy as np
import os, glob
from tqdm import tqdm
from models.pidnet_cl_p import PIDNetCL

# ==========================================
# 1. SETUP: RGB DATASET (Official Config)
# ==========================================
DATA_ROOT = "/content/drive/MyDrive/CSS/PIDNet/data/cityscapes"
NEW_CLASS_ID = 28
img_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # RGB Stats
])

class CityscapesCL(Dataset):
    def __init__(self, root, split="train"):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir = os.path.join(root, "gtFine", split)
        self.items = []
        # Find images
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                imgs = sorted(glob.glob(os.path.join(self.img_dir, city, "*_leftImg8bit.png")))
                for ip in imgs:
                    gp = os.path.join(self.gt_dir, city, os.path.basename(ip).replace("_leftImg8bit.png", "_gtFine_labelIds.png"))
                    if os.path.exists(gp): self.items.append((ip, gp))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        ip, gp = self.items[idx]
        # RGB Loading (Matches Official PIDNet training)
        img = img_tf(Image.open(ip).convert("RGB"))
        lbl = np.array(Image.open(gp), dtype=np.uint8)

        # Binary Target for Training (Bus vs Rest)
        tgt = np.full_like(lbl, 255)
        valid_bg = [7,8,11,12,13,17,19,20,21,22,23,24,25,26,27]
        for c in valid_bg: tgt[lbl == c] = 0
        tgt[lbl == NEW_CLASS_ID] = 1
        return img, torch.from_numpy(tgt).long(), lbl # Return raw label for eval

# ==========================================
# 2. MODEL: LOAD & FREEZE
# ==========================================
print("=> Initializing Model & Loading Base Weights...")
model = PIDNetCL(m=2, n=3, num_classes=15, planes=32, ppm_planes=96, head_planes=128, augment=True)
ckpt = torch.load("/content/drive/MyDrive/CSS/PIDNet/output/cityscapes/pidnet_s_custom_15/best.pt", map_location="cpu")
state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
state_dict = {k.replace('module.', '').replace('model.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)

# Freeze Old, Thaw New
for name, p in model.named_parameters():
    p.requires_grad = any(t in name for t in ["layer3_p_new", "layer4_p_new", "layer5_p_new", "seghead_new"])
model.cuda()
model.train()

# ==========================================
# 3. TRAIN: SUBSET ONLY
# ==========================================
print("=> Preparing Training Data...")
train_set = CityscapesCL(DATA_ROOT, "train")
bus_indices = [i for i, (_, gp) in enumerate(train_set.items) if (np.array(Image.open(gp)) == NEW_CLASS_ID).any()]
train_loader = DataLoader(Subset(train_set, bus_indices), batch_size=6, shuffle=True, num_workers=2, pin_memory=True)

print(f"=> Retraining New Branch (RGB) on {len(bus_indices)} images...")
optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
crit = nn.CrossEntropyLoss(ignore_index=255)
scaler = GradScaler(enabled=True)

for epoch in range(5): # Fast retraining (5 epochs of the subset is enough to recover)
    for i, (img, tgt, _) in enumerate(train_loader):
        img, tgt = img.cuda(), tgt.cuda()
        with autocast(enabled=True):
            out = model.forward_with_new(img)
            loss = crit(out["logits_new_2ch"], tgt)
        optim.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()
    print(f"   Epoch {epoch+1} Loss: {loss.item():.4f}")

# ==========================================
# 4. EVALUATE: DUAL MODE (LEGACY vs HONEST)
# ==========================================
print("\n=> Running Dual-Mode Evaluation (Validation Set)...")
val_set = CityscapesCL(DATA_ROOT, "val")
val_loader = DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)

@torch.no_grad()
def fuse(o15, onew, tau=0.55):
    p_new = torch.softmax(onew, dim=1)[:, 1:2]
    return torch.where(p_new >= tau, torch.tensor(15).to(p_new.device), o15.argmax(1, keepdim=True)).squeeze(1)

model.eval()
# Confusion Matrices
cm_legacy = torch.zeros((15, 15), dtype=torch.long).cuda() # Original 15 classes
cm_honest = torch.zeros((16, 16), dtype=torch.long).cuda() # All 16 classes

id_map_15 = {7:0,8:1,11:2,12:3,13:4,17:5,19:6,20:7,21:8,22:9,23:10,24:11,25:12,26:13,27:14}

for img, _, raw_lbl in tqdm(val_loader):
    img = img.cuda()
    raw_lbl = raw_lbl.long().cuda()

    # 1. Forward & Up
    out = model.forward_with_new(img)
    l15 = F.interpolate(out["logits_15"], raw_lbl.shape[-2:], mode='bilinear', align_corners=True)
    lnew = F.interpolate(out["logits_new_2ch"], raw_lbl.shape[-2:], mode='bilinear', align_corners=True)
    pred = fuse(l15, lnew, tau=0.55) # Use 0.55 or adjust higher if Truck drops too much

    # 2. Prepare Targets (16 Class)
    tgt_16 = torch.full_like(raw_lbl, 255)
    for k, v in id_map_15.items(): tgt_16[raw_lbl == k] = v
    tgt_16[raw_lbl == NEW_CLASS_ID] = 15

    # 3. Prepare Targets (15 Class Legacy - Ignore Bus)
    tgt_15 = tgt_16.clone()
    tgt_15[tgt_15 == 15] = 255 # IGNORE BUS PIXELS (Matches Base Model Evaluation)

    # 4. Update Matrices
    mask16 = tgt_16 != 255
    if mask16.any():
        cm_honest += torch.bincount(tgt_16[mask16]*16 + pred[mask16], minlength=256).view(16,16)

    mask15 = tgt_15 != 255
    if mask15.any():
        # Map predictions of "15" (Bus) back to something else or just mask them?
        # In legacy mode, if we predict "Bus", it's technically a "Non-15-class" prediction.
        # But usually we just calculate IoU on 0-14.
        p15_view = pred[mask15]
        # Only count if prediction is 0-14 (Valid Legacy)
        valid_p = p15_view < 15
        if valid_p.any():
            cm_legacy += torch.bincount(tgt_15[mask15][valid_p]*15 + p15_view[valid_p], minlength=225).view(15,15)

# Calculate Scores
iou_leg = torch.diag(cm_legacy) / (cm_legacy.sum(1) + cm_legacy.sum(0) - torch.diag(cm_legacy) + 1e-6)
iou_hon = torch.diag(cm_honest) / (cm_honest.sum(1) + cm_honest.sum(0) - torch.diag(cm_honest) + 1e-6)

print(f"\n15 Class mIoU: {iou_leg.mean().item()*100:.2f}%  -- Should be around 79%")
print(f"16 Class mIoU: {iou_hon.mean().item()*100:.2f}%  -- Real System Performance")

=> Initializing Model & Loading Base Weights...
=> Preparing Training Data...
=> Retraining New Branch (RGB) on 274 images...


/tmp/ipython-input-1055210800.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=True)
/tmp/ipython-input-1055210800.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):


   Epoch 1 Loss: 0.1520
   Epoch 2 Loss: 0.2833
   Epoch 3 Loss: 0.0772
   Epoch 4 Loss: 0.0393
   Epoch 5 Loss: 0.0638

=> Running Dual-Mode Evaluation (Validation Set)...


100%|██████████| 125/125 [03:12<00:00,  1.54s/it]


15 Class mIoU: 76.75%  -- Should be around 79%
16 Class mIoU: 67.97%  -- Real System Performance


I-Branch

In [ ]:
!cp -f /content/drive/MyDrive/CSS/model/pidnet_cl_i.py /content/PIDNet/models/pidnet_cl_i.py

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import GradScaler, autocast
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import numpy as np
import os, glob
from tqdm import tqdm
from models.pidnet_cl_i import PIDNetCL

# ==========================================
# 1. SETUP: RGB DATASET (Official Config)
# ==========================================
DATA_ROOT = "/content/drive/MyDrive/CSS/PIDNet/data/cityscapes"
NEW_CLASS_ID = 28
img_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # RGB Stats
])

class CityscapesCL(Dataset):
    def __init__(self, root, split="train"):
        self.img_dir = os.path.join(root, "leftImg8bit", split)
        self.gt_dir = os.path.join(root, "gtFine", split)
        self.items = []
        # Find images
        if os.path.exists(self.img_dir):
            for city in sorted(os.listdir(self.img_dir)):
                imgs = sorted(glob.glob(os.path.join(self.img_dir, city, "*_leftImg8bit.png")))
                for ip in imgs:
                    gp = os.path.join(self.gt_dir, city, os.path.basename(ip).replace("_leftImg8bit.png", "_gtFine_labelIds.png"))
                    if os.path.exists(gp): self.items.append((ip, gp))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        ip, gp = self.items[idx]
        # RGB Loading (Matches Official PIDNet training)
        img = img_tf(Image.open(ip).convert("RGB"))
        lbl = np.array(Image.open(gp), dtype=np.uint8)

        # Binary Target for Training (Bus vs Rest)
        tgt = np.full_like(lbl, 255)
        valid_bg = [7,8,11,12,13,17,19,20,21,22,23,24,25,26,27]
        for c in valid_bg: tgt[lbl == c] = 0
        tgt[lbl == NEW_CLASS_ID] = 1
        return img, torch.from_numpy(tgt).long(), lbl # Return raw label for eval

# ==========================================
# 2. MODEL: LOAD & FREEZE
# ==========================================
print("=> Initializing Model & Loading Base Weights...")
model = PIDNetCL(m=2, n=3, num_classes=15, planes=32, ppm_planes=96, head_planes=128, augment=True)
ckpt = torch.load("/content/drive/MyDrive/CSS/PIDNet/output/cityscapes/pidnet_s_custom_15/best.pt", map_location="cpu")
state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
state_dict = {k.replace('module.', '').replace('model.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)

# Freeze Old, Thaw New
for name, p in model.named_parameters():
    p.requires_grad = any(t in name for t in ["layer3_p_new", "layer4_p_new", "layer5_p_new", "seghead_new"])
model.cuda()
model.train()

# ==========================================
# 3. TRAIN: SUBSET ONLY
# ==========================================
print("=> Preparing Training Data...")
train_set = CityscapesCL(DATA_ROOT, "train")
bus_indices = [i for i, (_, gp) in enumerate(train_set.items) if (np.array(Image.open(gp)) == NEW_CLASS_ID).any()]
train_loader = DataLoader(Subset(train_set, bus_indices), batch_size=6, shuffle=True, num_workers=2, pin_memory=True)

print(f"=> Retraining New Branch (RGB) on {len(bus_indices)} images...")
optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
crit = nn.CrossEntropyLoss(ignore_index=255)
scaler = GradScaler(enabled=True)

for epoch in range(5): # Fast retraining (5 epochs of the subset is enough to recover)
    for i, (img, tgt, _) in enumerate(train_loader):
        img, tgt = img.cuda(), tgt.cuda()
        with autocast(enabled=True):
            out = model.forward_with_new(img)
            loss = crit(out["logits_new_2ch"], tgt)
        optim.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()
    print(f"   Epoch {epoch+1} Loss: {loss.item():.4f}")

# ==========================================
# 4. EVALUATE: DUAL MODE (LEGACY vs HONEST)
# ==========================================
print("\n=> Running Dual-Mode Evaluation (Validation Set)...")
val_set = CityscapesCL(DATA_ROOT, "val")
val_loader = DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)

@torch.no_grad()
def fuse(o15, onew, tau=0.55):
    p_new = torch.softmax(onew, dim=1)[:, 1:2]
    return torch.where(p_new >= tau, torch.tensor(15).to(p_new.device), o15.argmax(1, keepdim=True)).squeeze(1)

model.eval()
# Confusion Matrices
cm_legacy = torch.zeros((15, 15), dtype=torch.long).cuda() # Original 15 classes
cm_honest = torch.zeros((16, 16), dtype=torch.long).cuda() # All 16 classes

id_map_15 = {7:0,8:1,11:2,12:3,13:4,17:5,19:6,20:7,21:8,22:9,23:10,24:11,25:12,26:13,27:14}

for img, _, raw_lbl in tqdm(val_loader):
    img = img.cuda()
    raw_lbl = raw_lbl.long().cuda()

    # 1. Forward & Up
    out = model.forward_with_new(img)
    l15 = F.interpolate(out["logits_15"], raw_lbl.shape[-2:], mode='bilinear', align_corners=True)
    lnew = F.interpolate(out["logits_new_2ch"], raw_lbl.shape[-2:], mode='bilinear', align_corners=True)
    pred = fuse(l15, lnew, tau=0.55) # Use 0.55 or adjust higher if Truck drops too much

    # 2. Prepare Targets (16 Class)
    tgt_16 = torch.full_like(raw_lbl, 255)
    for k, v in id_map_15.items(): tgt_16[raw_lbl == k] = v
    tgt_16[raw_lbl == NEW_CLASS_ID] = 15

    # 3. Prepare Targets (15 Class Legacy - Ignore Bus)
    tgt_15 = tgt_16.clone()
    tgt_15[tgt_15 == 15] = 255 # IGNORE BUS PIXELS (Matches Base Model Evaluation)

    # 4. Update Matrices
    mask16 = tgt_16 != 255
    if mask16.any():
        cm_honest += torch.bincount(tgt_16[mask16]*16 + pred[mask16], minlength=256).view(16,16)

    mask15 = tgt_15 != 255
    if mask15.any():
        # Map predictions of "15" (Bus) back to something else or just mask them?
        # In legacy mode, if we predict "Bus", it's technically a "Non-15-class" prediction.
        # But usually we just calculate IoU on 0-14.
        p15_view = pred[mask15]
        # Only count if prediction is 0-14 (Valid Legacy)
        valid_p = p15_view < 15
        if valid_p.any():
            cm_legacy += torch.bincount(tgt_15[mask15][valid_p]*15 + p15_view[valid_p], minlength=225).view(15,15)

# Calculate Scores
iou_leg = torch.diag(cm_legacy) / (cm_legacy.sum(1) + cm_legacy.sum(0) - torch.diag(cm_legacy) + 1e-6)
iou_hon = torch.diag(cm_honest) / (cm_honest.sum(1) + cm_honest.sum(0) - torch.diag(cm_honest) + 1e-6)

print(f"\n15 Class mIoU: {iou_leg.mean().item()*100:.2f}%  -- Should be around 79%")
print(f"16 Class mIoU: {iou_hon.mean().item()*100:.2f}%  -- Real System Performance")

=> Initializing Model & Loading Base Weights...
=> Preparing Training Data...
=> Retraining New Branch (RGB) on 274 images...


/tmp/ipython-input-853363436.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=True)
/tmp/ipython-input-853363436.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=True):


   Epoch 1 Loss: 0.4119
   Epoch 2 Loss: 0.2379
   Epoch 3 Loss: 0.1275
   Epoch 4 Loss: 0.0667
   Epoch 5 Loss: 0.1346

=> Running Dual-Mode Evaluation (Validation Set)...


100%|██████████| 125/125 [00:42<00:00,  2.93it/s]


15 Class mIoU: 77.49%  -- Should be around 79%
16 Class mIoU: 70.25%  -- Real System Performance
